# TP-02: Data Cleaning with Python

In this notebook, we read a CSV file and clean its data. We solve the five exercises one by one, then use the classes together.

Run the cells from top to bottom. You need pandas. If it is missing, remove the `#` before `%pip install pandas` in the next cell and run that line once.


In [ ]:
# %pip install pandas

import pandas as pd
import time
from abc import ABC, abstractmethod
from functools import wraps
from pathlib import Path


## Create a small CSV file

We use five rows of student data. Some values are missing, some names have extra spaces, and one row is repeated.

The code creates a `data` folder inside `TP-02`. Running it again replaces the sample CSV.


In [ ]:
# Find the TP-02 folder.
folder = Path.cwd()
if (folder / "TP-02").is_dir():
    folder = folder / "TP-02"

folder = folder / "data"
folder.mkdir(exist_ok=True)

input_file = folder / "students.csv"
output_file = folder / "cleaned_students.csv"

students = pd.DataFrame({
    "Name": [" Dara ", "SOPHEA", " Dara ", "Vanna", "Sokha"],
    "Age": [20.0, 22.0, 20.0, None, 24.0],
    "City": [" Phnom Penh ", "SIEM REAP", " Phnom Penh ", None, "Siem Reap"],
    "Score": [80.0, None, 80.0, 60.0, 100.0]
})

students.to_csv(input_file, index=False)
print(students)


## Exercise 1: Create the CSVReader class

A class groups related data and functions together. Here, `self` means the current object.

- `__init__()` stores the file path.
- `read()` opens the CSV and stores its data in `self.df`.
- `preview(n)` prints the first `n` rows.


In [ ]:
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    def read(self):
        self.df = pd.read_csv(self.file_path)
        return self.df

    def preview(self, n=5):
        if self.df is None:
            self.read()
        print(self.df.head(n))


In [ ]:
# Create an object and use its methods.
reader = CSVReader(input_file)
df = reader.read()
reader.preview(3)


## Exercise 2: Handle missing values with strategies

A strategy is a way to do a task. We create three ways to handle missing values:

- `DropMissing`: remove rows with missing values.
- `FillMean`: fill missing numbers with the column average.
- `FillMode`: fill missing values with the most common value.

`ABC` and `@abstractmethod` make a base class that tells each child class to provide a `handle()` method. `DataCleaner` uses the strategy we give it.

We use `copy()` when changing values so the original data stays the same. A column with no known values stays missing because it has no average or most common value.


In [ ]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df):
        pass


class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna().copy()


class FillMean(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()
        number_columns = result.select_dtypes(include="number").columns

        for column in number_columns:
            average = result[column].mean()
            result[column] = result[column].fillna(average)

        return result


class FillMode(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()

        for column in result.columns:
            common_values = result[column].mode()
            if not common_values.empty:
                # Use the first value if several values are equally common.
                common_value = common_values.iloc[0]
                result[column] = result[column].fillna(common_value)

        return result


class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)


In [ ]:
# Try each strategy on the same original data.
cleaner = DataCleaner(DropMissing())
print("Drop rows:")
print(cleaner.clean(df))

cleaner = DataCleaner(FillMean())
print("\nFill with averages (text is still missing):")
print(cleaner.clean(df))

cleaner = DataCleaner(FillMode())
print("\nFill with most common values:")
print(cleaner.clean(df))


## Exercise 3: Add logging and timing decorators

A decorator adds extra steps before or after a function. We do not need to change the function's main code.

`*args` and `**kwargs` pass the inputs to the original function. `@wraps(func)` keeps its name. `log_action` prints start and finish messages, and `log_time` prints how long the function takes.

We apply both decorators to `DataCleaner.clean()` below. The decorators extend its behavior, which is the idea behind the Open/Closed Principle.


In [ ]:
def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Starting:", func.__name__)
        result = func(*args, **kwargs)
        print("Finished:", func.__name__)
        return result
    return wrapper


def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print("Time taken:", round(end - start, 6), "seconds")
        return result
    return wrapper


# This version adds decorators to the DataCleaner from Exercise 2.
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    @log_action
    @log_time
    def clean(self, df):
        return self.strategy.handle(df)


cleaner = DataCleaner(FillMean())
print(cleaner.clean(df))


## Exercise 4: Create transformations with a factory

A factory creates the object we ask for using a name. We use simple `if` and `elif` statements to choose the class.

- `NormalizeColumns` scales numbers between 0 and 1 using `(value - minimum) / (maximum - minimum)`. If all known numbers are equal, they become zero.
- `RemoveDuplicates` removes repeated rows.
- `StandardizeText` removes spaces at the ends and changes text to lowercase.

Each class has the same `apply()` method. This lets us use different transformations in the same way.


In [ ]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df):
        pass


class NormalizeColumns(DataTransform):
    def apply(self, df):
        result = df.copy()
        number_columns = result.select_dtypes(include="number").columns

        for column in number_columns:
            smallest = result[column].min()
            largest = result[column].max()

            if largest == smallest:
                # Multiplying by zero keeps missing values missing.
                result[column] = result[column] * 0
            else:
                result[column] = (result[column] - smallest) / (largest - smallest)

        return result


class RemoveDuplicates(DataTransform):
    def apply(self, df):
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    def apply(self, df):
        result = df.copy()
        text_columns = result.select_dtypes(include=["object", "string"]).columns

        for column in text_columns:
            result[column] = result[column].str.strip().str.lower()

        return result


class TransformFactory:
    @staticmethod
    def create(name):
        if name == "normalize":
            return NormalizeColumns()
        elif name == "duplicates":
            return RemoveDuplicates()
        elif name == "text":
            return StandardizeText()
        else:
            raise ValueError("Unknown transformation: " + name)


In [ ]:
# Ask the factory for each transformation.
transform = TransformFactory.create("normalize")
print("Normalize numbers:")
print(transform.apply(df))

transform = TransformFactory.create("duplicates")
print("\nRemove duplicates:")
print(transform.apply(df))

transform = TransformFactory.create("text")
print("\nStandardize text:")
print(transform.apply(df))


## Exercise 5: Build the full pipeline

A pipeline runs several steps in order. The Template Method Pattern puts this order in the base class:

1. Load the CSV.
2. Clean missing values.
3. Transform the data.
4. Save a new CSV.

`CSVDataPipeline` provides the code for each step. It uses the classes from the earlier exercises. In `clean()`, we tidy text and remove duplicates before calculating averages, so repeated rows do not affect the averages. We then fill numbers with averages and fill remaining text values with modes.


In [ ]:
class DataPipeline(ABC):
    def run(self):
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self.df

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self):
        pass

    @abstractmethod
    def transform(self):
        pass

    @abstractmethod
    def save(self):
        pass


class CSVDataPipeline(DataPipeline):
    def __init__(self, input_file, output_file):
        if Path(input_file).resolve() == Path(output_file).resolve():
            raise ValueError("Use different input and output files.")
        self.input_file = input_file
        self.output_file = output_file
        self.df = None

    def load(self):
        reader = CSVReader(self.input_file)
        self.df = reader.read()

    def clean(self):
        text_transform = TransformFactory.create("text")
        self.df = text_transform.apply(self.df)

        duplicate_transform = TransformFactory.create("duplicates")
        self.df = duplicate_transform.apply(self.df)

        mean_cleaner = DataCleaner(FillMean())
        self.df = mean_cleaner.clean(self.df)

        mode_cleaner = DataCleaner(FillMode())
        self.df = mode_cleaner.clean(self.df)

    def transform(self):
        number_transform = TransformFactory.create("normalize")
        self.df = number_transform.apply(self.df)

        # Filling missing values can sometimes create duplicate rows.
        duplicate_transform = TransformFactory.create("duplicates")
        self.df = duplicate_transform.apply(self.df)

    def save(self):
        self.df.to_csv(self.output_file, index=False)
        print("Saved to:", self.output_file)


In [ ]:
pipeline = CSVDataPipeline(input_file, output_file)
cleaned_df = pipeline.run()

print("\nFinal cleaned data:")
print(cleaned_df)


## Check the result

For this sample, we should have four rows, no missing values, and no duplicates. The names and cities should be lowercase, and the numbers should be between 0 and 1.

`assert` checks that a condition is true. If it is false, Python shows an error.


In [ ]:
print("Missing values in each column:")
print(cleaned_df.isna().sum())
print("Duplicate rows:", cleaned_df.duplicated().sum())

assert len(cleaned_df) == 4
assert cleaned_df.isna().sum().sum() == 0
assert cleaned_df.duplicated().sum() == 0
assert cleaned_df.loc[0, "Name"] == "dara"

for column in ["Age", "Score"]:
    assert cleaned_df[column].min() >= 0
    assert cleaned_df[column].max() <= 1

# Read the saved file to check it contains the cleaned data.
saved_df = pd.read_csv(output_file)
pd.testing.assert_frame_equal(cleaned_df, saved_df)
print("All checks passed!")
